# 01 · EDA and data quality
**Goal:** understand the target, find the data problems *before* modeling, and form hypotheses
that the later notebooks test. Sections: class balance, undocumented codes, outliers, signal.

In [ ]:
import sys, glob, warnings
sys.path.insert(0, "..")            # make `src` importable from notebooks/
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

from src import features as ft, evaluate as ev, train as tr, fairness as fr

DATA = sorted(glob.glob("../data/*.xls*") + glob.glob("../data/*.csv"))[0]   # see data/README.md
print("Using", DATA)

In [ ]:
raw = ft.load_raw(DATA)
print(raw.shape)
raw.head()

## 1. Class balance
About 22% of customers default. Accuracy would reward a model that says 'nobody defaults' (~78%), so we will use PR-AUC and cost-based metrics.

In [ ]:
rate = raw[ft.TARGET].mean()
print(f"Default rate: {rate:.1%}  |  always-predict-no-default accuracy: {1-rate:.1%}")
raw[ft.TARGET].value_counts().rename({0: "no default", 1: "default"}).plot.bar(rot=0, title="Class balance")
plt.ylabel("customers"); plt.savefig("../reports/figures/eda_class_balance.png", bbox_inches="tight")

## 2. Data quality: undocumented codes and oddities
The UCI documentation lists EDUCATION 1-4, MARRIAGE 1-3 and PAY_x as -1 or 1-9. The data contain other values. Flagging this is part of the job.

In [ ]:
ft.data_quality_report(raw)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
raw["EDUCATION"].value_counts().sort_index().plot.bar(ax=ax[0], title="EDUCATION (documented: 1-4)")
raw["MARRIAGE"].value_counts().sort_index().plot.bar(ax=ax[1], title="MARRIAGE (documented: 1-3)")
raw["PAY_1"].value_counts().sort_index().plot.bar(ax=ax[2], title="PAY_1 (documented: -1, 1-9)")
plt.tight_layout(); plt.savefig("../reports/figures/eda_undocumented_codes.png", bbox_inches="tight")

**Decisions** (documented, since a reviewer will ask):
* EDUCATION 0/5/6 → merged into 4 ("Other"); MARRIAGE 0 → 3 ("Other").
* PAY_x values -2 and 0 are undocumented. Their default rates (below) show they behave differently from -1,
  so we **keep them as distinct ordinal values** instead of collapsing them. Trees can use them directly and
  we add count features (`N_REVOLVING`, `N_NO_CONSUMPTION`, ...).
* Duplicate rows are dropped.

In [ ]:
df = ft.add_group_labels(ft.clean(raw))
# Default rate by repayment status: the strongest signal in this dataset
rate_by_pay = df.groupby("PAY_1")[ft.TARGET].agg(["mean", "size"]).rename(columns={"mean": "default_rate", "size": "n"})
display(rate_by_pay)
ax = rate_by_pay["default_rate"].plot.bar(rot=0, title="Default rate by last month's repayment status (PAY_1)")
ax.axhline(df[ft.TARGET].mean(), color="k", ls=":"); ax.set_ylabel("default rate")
plt.savefig("../reports/figures/eda_default_by_pay1.png", bbox_inches="tight")

## 3. Outliers and heavy tails
Bill and payment amounts are heavily right-skewed, include negative bills (credit balances) and over-limit statements. Trees are insensitive to this; logistic regression is not, which is one reason to build ratio features.

In [ ]:
cols = ["LIMIT_BAL", "BILL_AMT1", "PAY_AMT1"]
print(df[cols + ["AGE"]].describe(percentiles=[.01, .5, .99]).round(0))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for a, c in zip(ax, cols):
    np.log10(df[c].clip(lower=1)).hist(bins=50, ax=a); a.set_title(f"log10({c})")
plt.tight_layout(); plt.savefig("../reports/figures/eda_skew.png", bbox_inches="tight")
print("Negative bills:", int((df[ft.BILL_COLS] < 0).any(axis=1).sum()),
      "| statements above limit:", int(df[ft.BILL_COLS].gt(df.LIMIT_BAL, axis=0).any(axis=1).sum()))

## 4. Signal: payment behavior vs. demographics
Behavioral features should dominate. Demographics should be weaker, which matters for the fairness section (we want to be able to drop SEX/MARRIAGE at little cost).

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
df.groupby("SEX_LABEL")[ft.TARGET].mean().plot.bar(ax=ax[0, 0], rot=0, title="Default rate by sex")
df.groupby("EDU_LABEL")[ft.TARGET].mean().plot.bar(ax=ax[0, 1], rot=0, title="By education")
df.groupby("AGE_GROUP")[ft.TARGET].mean().reindex(["<30", "30-39", "40-49", "50-59", "60+"]).plot.bar(ax=ax[1, 0], rot=0, title="By age group")
df.groupby(pd.qcut(df.LIMIT_BAL, 10, duplicates="drop"), observed=True)[ft.TARGET].mean().plot.bar(ax=ax[1, 1], title="By credit-limit decile")
for a in ax.ravel(): a.set_xlabel(""); a.axhline(df[ft.TARGET].mean(), color="k", ls=":")
plt.tight_layout(); plt.savefig("../reports/figures/eda_default_by_group.png", bbox_inches="tight")

In [ ]:
# Engineered features: do they separate the classes better than the raw ones?
Xe, y = ft.get_xy(df)
corr = Xe.corrwith(y).sort_values(key=abs, ascending=False)
corr.head(12).to_frame("corr_with_default")

## Takeaways to write up
* Class balance & why accuracy is the wrong metric.
* The undocumented codes and how you resolved each.
* Which signals dominate (expect recent repayment status), and how strong the demographic signal is.
* Anything surprising (e.g. PAY = -2 / 0 behave differently from -1).